# Urban Heat & Cooling-Priority Mapping — Sensitivity Pillar (SingStat)

**NUS-ISS Practice Module, Week 2.** Builds `sensitivity_pillar.csv` for
`rank_impact.ipynb` from real SingStat population/elderly data — no more
`TOY_MODE`. One notebook, run top to bottom:

1. **Setup** — install deps, mount Drive (no Earth Engine needed here)
2. **SP.1** — config
3. **SP.2** — fetch SingStat population-by-subzone data from data.gov.sg
4. **SP.3** — clean + filter (drop planning-area totals, handle suppressed cells)
5. **SP.4** — compute population_total / elderly_proportion per subzone
6. **SP.5** — match subzone names to your heat-variants CSV's subzone_id
7. **SP.6** — combine into sensitivity_raw (⚠️ placeholder formula — see SP.6)
8. **SP.7** — verdict
9. **SP.8** — save to Drive

Run cells in order.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [3]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q requests pandas


## Setup 2 — Mount Google Drive

In [4]:
# --- SETUP CELL 2: Mount Google Drive ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 3 — Initialize shared results tracker

In [5]:
# --- SETUP CELL 3: Initialize shared results tracker ------------------------
sp_results = {}
print("sp_results initialized — populated by the SP.7 verdict cell.")


sp_results initialized — populated by the SP.7 verdict cell.


---
# SP — Sensitivity Pillar (population + elderly, per subzone)


## SP.1 — Config

`HEAT_CSV_PATH` must point at the same file `rank_impact.ipynb` reads —
matching happens against the `subzone_id` values actually in that file.


In [6]:
# --- SP CELL 1: Config -------------------------------------------------------
POP_DATASET_ID = "d_d95ae740c0f8961a0b10435836660ce0"  # SingStat: Resident Population by
                                                          # Planning Area/Subzone, Age Group
                                                          # and Sex (Census of Population 2020),
                                                          # basis = URA Master Plan 2019 — same
                                                          # basis as your subzones GeoJSON.

EXPORT_FOLDER = "urban_heat_sg"
HEAT_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/heat_variants_subzone.csv"
OUT_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/sensitivity_pillar.csv"

ELDERLY_AGE_COLUMNS = ["Total_65_69", "Total_70_74", "Total_75_79", "Total_80_84", "Total_85_89", "Total_90andOver"]
TOTAL_POP_COLUMN = "Total_Total"
NAME_COLUMN = "Number"  # this dataset's subzone/planning-area name field is literally called "Number"

import os
if not os.path.exists(HEAT_CSV_PATH):
    raise FileNotFoundError(
        f"{HEAT_CSV_PATH} not found. Run gee_heat_variants.ipynb's export first, "
        f"then re-run this cell."
    )
print("✅ Heat-variants CSV found — will match subzone_id against it in SP.5.")


✅ Heat-variants CSV found — will match subzone_id against it in SP.5.


## SP.2 — Fetch SingStat population data

Uses the CKAN `datastore_search` endpoint (same family as your other
data.gov.sg pulls, different resource type than the GeoJSON poll-download).
Paginates explicitly rather than trusting a single call to return everything
— CKAN's default page size is small enough that a ~390-row table can get
silently truncated if you don't loop.


In [7]:
# --- SP CELL 2: Fetch SingStat population data (paginated) ------------------
import requests
import pandas as pd

def fetch_datastore_all(resource_id, page_size=500):
    base_url = "https://data.gov.sg/api/action/datastore_search"
    records = []
    offset = 0
    while True:
        r = requests.get(base_url, params={"resource_id": resource_id, "limit": page_size, "offset": offset})
        r.raise_for_status()
        payload = r.json()
        if not payload.get("success"):
            raise RuntimeError(f"data.gov.sg API error: {payload}")
        batch = payload["result"]["records"]
        records.extend(batch)
        print(f"  fetched {len(batch)} records (offset {offset}, total so far {len(records)})")
        if len(batch) < page_size:
            break
        offset += page_size
    return records

records = fetch_datastore_all(POP_DATASET_ID)
pop_raw = pd.DataFrame(records)
print(f"\nTotal records fetched: {len(pop_raw)}")
print("Columns:", list(pop_raw.columns))
pop_raw.head()


  fetched 388 records (offset 0, total so far 388)

Total records fetched: 388
Columns: ['_id', 'Number', 'Total_Total', 'Total_0_4', 'Total_5_9', 'Total_10_14', 'Total_15_19', 'Total_20_24', 'Total_25_29', 'Total_30_34', 'Total_35_39', 'Total_40_44', 'Total_45_49', 'Total_50_54', 'Total_55_59', 'Total_60_64', 'Total_65_69', 'Total_70_74', 'Total_75_79', 'Total_80_84', 'Total_85_89', 'Total_90andOver', 'Males_Total', 'Males_0_4', 'Males_5_9', 'Males_10_14', 'Males_15_19', 'Males_20_24', 'Males_25_29', 'Males_30_34', 'Males_35_39', 'Males_40_44', 'Males_45_49', 'Males_50_54', 'Males_55_59', 'Males_60_64', 'Males_65_69', 'Males_70_74', 'Males_75_79', 'Males_80_84', 'Males_85_89', 'Males_90andOver', 'Females_Total', 'Females_0_4', 'Females_5_9', 'Females_10_14', 'Females_15_19', 'Females_20_24', 'Females_25_29', 'Females_30_34', 'Females_35_39', 'Females_40_44', 'Females_45_49', 'Females_50_54', 'Females_55_59', 'Females_60_64', 'Females_65_69', 'Females_70_74', 'Females_75_79', 'Females_

,_id,Number,Total_Total,Total_0_4,Total_5_9,Total_10_14,Total_15_19,Total_20_24,Total_25_29,Total_30_34,...,Females_45_49,Females_50_54,Females_55_59,Females_60_64,Females_65_69,Females_70_74,Females_75_79,Females_80_84,Females_85_89,Females_90andOver
0,1,Total,4044210,183080,198740,206390,215230,244540,287000,297800,...,160050,150690,152870,143160,116790,89190,50220,38630,23060,14560
1,2,Ang Mo Kio - Total,162280,5280,6100,7030,7600,8680,10320,10490,...,6500,6140,6450,6650,6480,5350,3200,2350,1390,810
2,3,Ang Mo Kio Town Centre,4810,170,240,280,320,270,280,290,...,250,180,150,160,140,130,80,60,30,20
3,4,Cheng San,28070,1060,1040,1040,1160,1330,1710,2000,...,1140,1050,1060,1170,1210,920,540,360,210,130
4,5,Chong Boon,26500,860,840,1010,1060,1310,1610,1890,...,940,990,1040,1110,1130,990,610,440,250,140


## SP.3 — Clean + filter

Drops the Singapore-wide `"Total"` row and every planning-area subtotal row
(pattern: `"<Planning Area> - Total"`), since those aren't subzones and would
silently corrupt a subzone-level join if left in. Suppressed/small cells are
the literal string `"-"` in this dataset — converted to 0, which undercounts
truly tiny subzones (there are some, e.g. near-zero-population industrial
areas) but that's a defensible default here given rounding-to-nearest-10 in
the source and the alternative (dropping the row) loses the subzone entirely.


In [8]:
# --- SP CELL 3: Clean + filter -----------------------------------------------
pop = pop_raw.copy()

is_pa_total = pop[NAME_COLUMN].str.contains(" - Total", regex=False, na=False)
is_grand_total = pop[NAME_COLUMN] == "Total"
n_pa_total = is_pa_total.sum()
n_grand_total = is_grand_total.sum()

pop = pop[~is_pa_total & ~is_grand_total].copy()
print(f"Dropped {n_pa_total} planning-area total rows and {n_grand_total} grand-total row.")
print(f"Remaining (subzone-level) rows: {len(pop)}")

numeric_cols = [TOTAL_POP_COLUMN] + ELDERLY_AGE_COLUMNS
for col in numeric_cols:
    pop[col] = pop[col].replace("-", "0")
    pop[col] = pd.to_numeric(pop[col], errors="coerce")

n_nan = pop[numeric_cols].isna().any(axis=1).sum()
if n_nan:
    print(f"⚠️  {n_nan} rows have non-numeric values outside the expected '-' pattern after coercion — inspect before trusting downstream numbers.")
else:
    print("✅ All numeric columns parsed cleanly.")

zero_pop_rows = (pop[TOTAL_POP_COLUMN] == 0).sum()
if zero_pop_rows:
    print(f"ℹ️  {zero_pop_rows} subzones have zero recorded population (e.g. industrial/port areas) — "
          f"elderly_proportion will be 0/0 for these, handled as NaN in SP.4, not silently zero.")


Dropped 54 planning-area total rows and 1 grand-total row.
Remaining (subzone-level) rows: 333
✅ All numeric columns parsed cleanly.
ℹ️  46 subzones have zero recorded population (e.g. industrial/port areas) — elderly_proportion will be 0/0 for these, handled as NaN in SP.4, not silently zero.


## SP.4 — Compute population_total / elderly_proportion

In [9]:
# --- SP CELL 4: Compute pillars -----------------------------------------------
pop["population_total"] = pop[TOTAL_POP_COLUMN]
pop["elderly_total"] = pop[ELDERLY_AGE_COLUMNS].sum(axis=1)
pop["elderly_proportion"] = pop["elderly_total"] / pop["population_total"]
# 0/0 -> NaN (zero-population subzones), not 0 — a 0.0 elderly_proportion would
# falsely read as "no elderly", when the correct reading is "no residents at all".

print(pop[[NAME_COLUMN, "population_total", "elderly_total", "elderly_proportion"]].describe())
pop[[NAME_COLUMN, "population_total", "elderly_total", "elderly_proportion"]].head(10)


       population_total  elderly_total  elderly_proportion
count        333.000000     333.000000          287.000000
mean       12150.720721    1847.117117            0.135750
std        17763.252886    2613.147615            0.096217
min            0.000000       0.000000            0.000000
25%           30.000000       0.000000            0.072825
50%         4200.000000     650.000000            0.144330
75%        16920.000000    3120.000000            0.195443
max       130980.000000   20200.000000            0.863636


,Number,population_total,elderly_total,elderly_proportion
2,Ang Mo Kio Town Centre,4810,780,0.162162
3,Cheng San,28070,6030,0.214820
4,Chong Boon,26500,6440,0.243019
5,Kebun Bahru,22620,5080,0.224580
6,Sembawang Hills,6850,1320,0.192701
7,Shangri-La,15960,3570,0.223684
8,Tagore,7950,1510,0.189937
9,Townsville,21140,5060,0.239357
10,Yio Chu Kang,20,0,0.000000
11,Yio Chu Kang East,4200,720,0.171429


## SP.5 — Match subzone names to heat-variants `subzone_id`

Case-insensitive, whitespace-stripped match — this dataset and your URA
GeoJSON are both on Master Plan 2019, so names should mostly align, but
punctuation/spacing differences are common enough to check explicitly rather
than assume a clean join.


In [10]:
# --- SP CELL 5: Match to subzone_id ------------------------------------------
heat = pd.read_csv(HEAT_CSV_PATH)
heat_ids = heat["subzone_id"].astype(str)

def norm(s):
    return s.strip().upper()

heat_lookup = {norm(s): s for s in heat_ids}
pop["_name_norm"] = pop[NAME_COLUMN].apply(norm)
pop["subzone_id"] = pop["_name_norm"].map(heat_lookup)

n_matched = pop["subzone_id"].notna().sum()
n_unmatched_pop = pop["subzone_id"].isna().sum()
matched_heat_ids = set(pop["subzone_id"].dropna())
n_unmatched_heat = len(set(heat_ids) - matched_heat_ids)

print(f"Matched: {n_matched} / {len(pop)} SingStat subzone rows")
print(f"Unmatched SingStat rows (in population data, no match in heat CSV): {n_unmatched_pop}")
print(f"Unmatched heat-CSV subzones (no population match — will be missing from sensitivity_pillar.csv): {n_unmatched_heat}")

if n_unmatched_pop:
    print("\nSample unmatched SingStat names (check spelling/punctuation against your GeoJSON):")
    print(pop.loc[pop["subzone_id"].isna(), NAME_COLUMN].head(15).to_string(index=False))

if n_unmatched_heat:
    unmatched_heat_names = sorted(set(heat_ids) - matched_heat_ids)
    print("\nSample heat-CSV subzones with no population match:")
    for name in unmatched_heat_names[:15]:
        print(f"  {name}")
    print("\n⚠️  These subzones will be dropped from sensitivity_pillar.csv and will fail")
    print("   the join in rank_impact.ipynb's RI.2 (it will report the drop count) unless fixed here.")


Matched: 332 / 333 SingStat subzone rows
Unmatched SingStat rows (in population data, no match in heat CSV): 1
Unmatched heat-CSV subzones (no population match — will be missing from sensitivity_pillar.csv): 0

Sample unmatched SingStat names (check spelling/punctuation against your GeoJSON):
Changi- Total


## SP.6 — Combine into `sensitivity_raw`

⚠️ **This is a placeholder combination, not a locked S6 decision.** Right now
it's `0.5 * normalize(population_total) + 0.5 * normalize(elderly_proportion)`
— an even split between "how many people" and "what fraction are elderly."
That 50/50 split, and even whether population_total should be raw count vs.
density (population ÷ subzone area), is a call for whoever owns S6's
Sensitivity pillar definition. Change the formula below once that's decided;
don't treat this default as final.


In [11]:
# --- SP CELL 6: Combine into sensitivity_raw (PLACEHOLDER FORMULA) ----------
matched = pop[pop["subzone_id"].notna()].copy()

def normalize(s):
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

matched["elderly_proportion_filled"] = matched["elderly_proportion"].fillna(0)
# zero-population subzones get elderly_proportion=NaN from SP.4; filled to 0 only
# for this combine step so they don't propagate NaN into sensitivity_raw — they'll
# still show population_total=0, so they land at the bottom of any population-driven
# ranking regardless.

matched["sensitivity_raw"] = (
    0.5 * normalize(matched["population_total"])
    + 0.5 * normalize(matched["elderly_proportion_filled"])
)

print("Sensitivity pillar (top 10 by sensitivity_raw):")
print(matched[["subzone_id", "population_total", "elderly_proportion", "sensitivity_raw"]]
      .sort_values("sensitivity_raw", ascending=False)
      .head(10).to_string(index=False))


Sensitivity pillar (top 10 by sensitivity_raw):
         subzone_id  population_total  elderly_proportion  sensitivity_raw
      TAMPINES EAST            130980            0.154222         0.589286
        LOYANG WEST               220            0.863636         0.500840
        BEDOK NORTH             81840            0.204057         0.430552
     WOODLANDS EAST             98980            0.082845         0.425807
      TAMPINES WEST             79670            0.140831         0.385664
             YUNNAN             67500            0.122370         0.328519
           HONG KAH             53890            0.174429         0.306704
        BEDOK SOUTH             46690            0.218248         0.304587
JURONG WEST CENTRAL             63760            0.104454         0.303869
        YISHUN WEST             53910            0.162864         0.300084


## SP.7 — Verdict

In [12]:
# --- SP CELL 7: Verdict -------------------------------------------------------
print("\n--- SP Verdict ---")

sp_checks = {
    "Population data fetched (non-zero rows)": len(pop_raw) > 0,
    "Planning-area totals dropped": n_pa_total > 0,
    "Numeric columns parsed cleanly": n_nan == 0,
    "Majority of heat-CSV subzones matched (>=90%)": (len(heat_ids) - n_unmatched_heat) >= 0.9 * len(heat_ids),
    "sensitivity_raw computed for all matched rows": matched["sensitivity_raw"].notna().all(),
}

for check, passed in sp_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

sp_pass = all(sp_checks.values())
if sp_pass:
    print("\n✅ SP PASS: sensitivity_pillar.csv ready to save.")
else:
    print("\n⚠️  SP FAIL/MARGINAL: resolve flagged step(s) above — especially unmatched")
    print("   subzone names in SP.5 — before this feeds a real rank_impact.ipynb run.")

sp_results["SP_sensitivity_pillar"] = {
    "status": "PASS" if sp_pass else "FAIL",
    "checks": sp_checks,
    "n_subzones_matched": int(n_matched),
    "n_heat_subzones_unmatched": int(n_unmatched_heat),
}



--- SP Verdict ---
  [PASS] Population data fetched (non-zero rows)
  [PASS] Planning-area totals dropped
  [PASS] Numeric columns parsed cleanly
  [PASS] Majority of heat-CSV subzones matched (>=90%)
  [PASS] sensitivity_raw computed for all matched rows

✅ SP PASS: sensitivity_pillar.csv ready to save.


## SP.8 — Save to Drive

In [13]:
# --- SP CELL 8: Save to Drive --------------------------------------------------
out = matched[["subzone_id", "population_total", "elderly_proportion", "sensitivity_raw"]]
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} ({len(out)} subzones)")
print("\nPoint SENSITIVITY_CSV_PATH at this file in rank_impact.ipynb's RI.1, "
      "and set TOY_MODE = False once adaptive_capacity_pillar.csv is also ready.")


Saved: /content/drive/MyDrive/urban_heat_sg/sensitivity_pillar.csv (332 subzones)

Point SENSITIVITY_CSV_PATH at this file in rank_impact.ipynb's RI.1, and set TOY_MODE = False once adaptive_capacity_pillar.csv is also ready.
